# A-2 · Seasonality: what "normal" looks like (Q2)
**Stream A, Person A** · feeds slide 4 · method from `docs/who_answers_what.md`

**Why this matters:** every other stream compares periods. If they compare a December with a June, or a period that includes Black Friday with one that doesn't, seasonality will look like a program effect. This notebook measures the seasonal pattern, so the team can control for it.

**Questions**
1. How much does revenue swing by calendar month, store-wide and by category?
2. Do outerwear and hoodies peak in Nov–Jan? When do summer categories peak?
3. How big are Black Friday weekend and Boxing Week (the only discounts, per the case)?

**Method**
- Purchases only (`row_type == C.PURCHASE`). The data runs Nov 2022 – Oct 2026, so **every calendar month appears exactly 4 times**, which keeps the averages balanced.
- **Seasonal index** = average revenue in that calendar month ÷ that row's average month × 100. So 100 = a normal month, 200 = double.
- **Promo flag:** price paid vs. list price. 25% off = Black Friday weekend, 30% off = Boxing Week (the case's only discounts).

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.colors import TwoSlopeNorm

from src.metrics import load
from src import config as C

OUT = C.RESULTS_DIR / "a_competitor"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "legend.frameon": False})
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

lines = load("lines")
p = lines[lines.row_type == C.PURCHASE].copy()
p["cal_month"] = p.date.dt.month

# Promo flag from the discount actually given (computed here because src/ is frozen)
p["discount"] = (1 - p.unit_price_paid / p.price).round(2)
p["promo"] = p.discount.map({0.25: "Black Friday", 0.30: "Boxing Week"}).fillna("Full price")
assert set(p.discount.unique()) <= {0.0, 0.25, 0.30}, "unexpected discount level"

n_years = p.groupby("cal_month").date.apply(lambda d: d.dt.year.nunique())
assert (n_years == 4).all(), "each calendar month should appear 4 times"
print(p.promo.value_counts().to_string())

## 1. Store-wide seasonal pattern

In [ ]:
store = p.groupby("cal_month").revenue.sum() / 4
store_idx = (store / store.mean() * 100).rename(index=dict(enumerate(MONTHS, 1)))

# Robustness: the 4-year average includes the competitor dip (Mar 2024 - Feb 2025).
# Compare with the one clean pre-competitor year (Mar 2023 - Feb 2024).
pre = p[(p.date >= "2023-03-01") & (p.date < "2024-03-01")].groupby("cal_month").revenue.sum()
pre_idx = (pre / pre.mean() * 100).rename(index=dict(enumerate(MONTHS, 1)))
seasonal = pd.DataFrame({"All 4 years (Nov 2022 – Oct 2026)": store_idx,
                         "Pre-competitor year (Mar 2023 – Feb 2024)": pre_idx}).T
seasonal.columns.name = None
print(f"Correlation between the two profiles: {store_idx.corr(pre_idx):.2f}")
(seasonal.style.format("{:.0f}")
 .set_caption("<b>Store-wide seasonal index by calendar month</b> (100 = an average month). "
              "Example: Dec 194 = December brings in ~1.9× an average month. "
              "The second row is a check that the competitor years didn't distort the pattern.")
 .set_table_styles([{"selector": "caption", "props": "caption-side: top; text-align: left; font-size: 0.95em;"}]))

### 1b. Robustness: does the way we calculate the index matter?
The simple index above assumes there's **no long-term trend**. Our data has one: a competitor dip in 2024, then growth in 2025–26. It also ignores that months have different numbers of days. So we compare four methods:

| Method | How it works | Trade-off |
|---|---|---|
| **Average month** (used above) | Average revenue per calendar month ÷ average of all months | Simplest to explain. Ignores trend and month length |
| **Per open day** | Revenue per day the store was open (closed Dec 25), then the same index | Fixes February's 28 days |
| **Ratio to moving average** | Each month ÷ a centred 12-month moving average, which removes the trend, then averaged by calendar month | Removes trend. Loses 6 months at each end of the data |
| **Regression** | log(monthly revenue) on month dummies + a linear trend + a flag for the competitor year (Mar 2024 – Feb 2025) | Controls for trend **and** the competitor shock at once |

If all four agree, the seasonal pattern doesn't depend on the method.

In [ ]:
m = p.groupby(p.date.dt.to_period("M")).revenue.sum()          # 48 monthly totals
cal = m.index.month

# 1. average month (same as above)
avg = m.groupby(cal).mean()
idx_avg = avg / avg.mean() * 100

# 2. per open day (store closed on Dec 25)
open_days = pd.Series([per.days_in_month - (per.month == 12) for per in m.index], index=m.index)
pdm = (m / open_days).groupby(cal).mean()
idx_day = pdm / pdm.mean() * 100

# 3. ratio to a centred 2x12 moving average (classical decomposition)
ma = m.rolling(12, center=True).mean()
ma = (ma + ma.shift(-1)) / 2
ratio = (m / ma).dropna()
rma = ratio.groupby(ratio.index.month).mean()
idx_ma = rma / rma.mean() * 100

# 4. regression: log(revenue) ~ month dummies + trend + competitor-year flag (least squares, numpy only)
comp_year = (m.index >= C.COMPETITOR_OPEN.to_period("M")) & (m.index < (C.COMPETITOR_OPEN + pd.DateOffset(years=1)).to_period("M"))
X = pd.get_dummies(pd.Series(cal, name="m"), prefix="m", drop_first=True).astype(float)
X.insert(0, "const", 1.0)
X["trend"] = np.arange(len(m))
X["competitor_year"] = comp_year.astype(float)
beta, *_ = np.linalg.lstsq(X.values, np.log(m.values), rcond=None)
b = dict(zip(X.columns, beta))
month_eff = np.exp([0.0] + [b[f"m_{k}"] for k in range(2, 13)])
idx_reg = pd.Series(month_eff / month_eff.mean() * 100, index=range(1, 13))

methods = pd.DataFrame({
    "Average month (used)": idx_avg.values,
    "Per open day": idx_day.values,
    "Ratio to moving average": idx_ma.values,
    "Regression (trend + competitor)": idx_reg.values,
}, index=MONTHS).T
max_gap = (methods - methods.iloc[0]).abs().max().max()

print(f"Ratio-to-moving-average uses {len(ratio)} of {len(m)} months ({ratio.index.min()} to {ratio.index.max()})")
print(f"Largest difference from the simple index: {max_gap:.0f} points "
      f"(in {(methods - methods.iloc[0]).abs().max().idxmax()})")
first_year = m[comp_year].sum() / m[(m.index >= pd.Period("2023-03", "M")) & (m.index <= pd.Period("2024-02", "M"))].sum() - 1
print(f"Regression side result: competitor year = {np.exp(b['competitor_year']) - 1:+.0%} revenue, "
      f"after controlling for season and trend (simple comparison, Mar 2024–Feb 2025 vs the year before: {first_year:+.0%})")
print(f"Underlying trend: {np.exp(b['trend'] * 12) - 1:+.1%} per year")

(methods.style.format("{:.0f}")
 .set_caption("<b>Seasonal index by method</b> (100 = an average month). All four methods agree within "
              f"{max_gap:.0f} points, so the seasonal pattern is robust.")
 .set_table_styles([{"selector": "caption", "props": "caption-side: top; text-align: left; font-size: 0.95em;"}]))

## 2. Category × calendar month heatmap

In [ ]:
cat = p.groupby(["category", "cal_month"]).revenue.sum().unstack() / 4
cat_idx = cat.div(cat.mean(axis=1), axis=0) * 100
# order rows: winter categories, year-round, summer (by peak month)
order = ["Outerwear", "Hoodies & Sweaters", "Tops", "Leggings", "Accessories", "Sports Bras", "Tank Tops", "Shorts"]
cat_idx = cat_idx.loc[order]
heat = pd.concat([cat_idx, store_idx.set_axis(range(1, 13)).to_frame("Whole store").T])

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(heat.values, cmap="RdBu_r", norm=TwoSlopeNorm(vcenter=100, vmin=25, vmax=300), aspect="auto")
ax.set_xticks(range(12), MONTHS)
ax.set_yticks(range(len(heat)), heat.index)
ax.axhline(len(heat) - 1.5, color="white", lw=3)        # separate the store row
for i in range(heat.shape[0]):
    for j in range(12):
        v = heat.iat[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8.5,
                color="white" if (v > 200 or v < 45) else "#222",
                fontweight="bold" if i == len(heat) - 1 else "normal")
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
cb.set_label("100 = an average month for that row", fontsize=8.5)
ax.set_title("December is ~2× a normal month; winter gear peaks at ~3× in Dec, summer gear in May–Aug", fontsize=11.5)
ax.set_xlabel("Seasonal index = average revenue in that calendar month ÷ the row's average month × 100 "
              "(Nov 2022 – Oct 2026, each month counted 4 times)", fontsize=8.5, color="#444")
plt.tight_layout(); fig.savefig(OUT / "a2_seasonality_heatmap.png", bbox_inches="tight"); plt.show()

In [ ]:
peaks = pd.DataFrame({
    "peak month": cat_idx.idxmax(axis=1).map(dict(enumerate(MONTHS, 1))),
    "peak index": cat_idx.max(axis=1).round(0),
    "low month": cat_idx.idxmin(axis=1).map(dict(enumerate(MONTHS, 1))),
    "low index": cat_idx.min(axis=1).round(0),
    "Nov–Jan share of year": cat[[11, 12, 1]].sum(axis=1) / cat.sum(axis=1),
})
peaks.style.format({"Nov–Jan share of year": "{:.0%}", "peak index": "{:.0f}", "low index": "{:.0f}"})

## 3. How big are Black Friday weekend and Boxing Week?
Measured per holiday season (Nov–Dec of each year). The discount given is list price minus price paid, which is margin handed back to customers.

In [ ]:
p["season_year"] = np.where(p.cal_month >= 11, p.date.dt.year, p.date.dt.year - 1)   # Nov-Oct "retail year"
rows = []
for y in sorted(p.season_year.unique()):
    yr = p[p.season_year == y]
    nd = yr[yr.cal_month.isin([11, 12])]
    days = nd.groupby("promo").date.nunique()
    rev = nd.groupby("promo").revenue.sum()
    disc = (nd.price * nd.quantity - nd.revenue).groupby(nd.promo).sum()
    rows.append({
        "retail year": f"Nov {y} – Oct {y + 1}",
        "BF days": days.get("Black Friday", 0), "BW days": days.get("Boxing Week", 0),
        "BF revenue": rev.get("Black Friday", 0), "BW revenue": rev.get("Boxing Week", 0),
        "promo share of year": (rev.get("Black Friday", 0) + rev.get("Boxing Week", 0)) / yr.revenue.sum(),
        "Nov–Dec share of year": nd.revenue.sum() / yr.revenue.sum(),
        "discount given": disc.get("Black Friday", 0) + disc.get("Boxing Week", 0),
        "promo $/day ÷ other Nov–Dec $/day": (rev.drop("Full price").sum() / days.drop("Full price").sum())
                                            / (rev["Full price"] / days["Full price"]),
    })
promo = pd.DataFrame(rows).set_index("retail year")
promo.style.format({"BF revenue": "${:,.0f}", "BW revenue": "${:,.0f}", "discount given": "${:,.0f}",
                    "promo share of year": "{:.1%}", "Nov–Dec share of year": "{:.0%}",
                    "promo $/day ÷ other Nov–Dec $/day": "{:.2f}×"})

In [ ]:
daily = p.groupby("date").agg(revenue=("revenue", "sum"),
                              promo=("promo", lambda s: s[s != "Full price"].iloc[0] if (s != "Full price").any() else "Full price"))
daily["bucket"] = np.select(
    [daily.promo == "Black Friday", daily.promo == "Boxing Week", daily.index.month.isin([11, 12])],
    ["Black Friday weekend", "Boxing Week", "Other Nov–Dec days"], default="Rest of year (Jan–Oct)")
per_day = daily.groupby("bucket").revenue.mean().reindex(
    ["Rest of year (Jan–Oct)", "Other Nov–Dec days", "Boxing Week", "Black Friday weekend"])

fig, ax = plt.subplots(figsize=(9, 3.6))
colors = ["#9AA5B1", "#5B8DB8", "#D1495B", "#D1495B"]
ax.barh(per_day.index, per_day.values, color=colors)
base = per_day.iloc[0]
for i, v in enumerate(per_day.values):
    ax.text(v + 400, i, f"${v/1e3:,.1f}K/day  ({v / base:.1f}× a normal day)", va="center", fontsize=9)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax.set(title="Sale days sell ~2.5× a normal day, but only ~1.4× other holiday-season days",
       xlabel="Average purchase revenue per day (Nov 2022 – Oct 2026)", xlim=(0, per_day.max() * 1.45))
ax.grid(axis="x", alpha=0.25)
plt.tight_layout(); fig.savefig(OUT / "a2_promo_days.png", bbox_inches="tight"); plt.show()

## 4. Headline numbers to hand over

In [ ]:
summary = {
    "December vs. average month (store)": f"{store_idx['Dec']:.0f} (≈{store_idx['Dec'] / 100:.1f}×)",
    "Quietest months (store)": ", ".join(f"{m} {v:.0f}" for m, v in store_idx.nsmallest(3).items()),
    "Sep–Oct (back to school / fall)": f"{store_idx['Sep']:.0f} / {store_idx['Oct']:.0f}",
    "Outerwear: Dec peak / Apr–Aug low": f"{cat_idx.loc['Outerwear', 12]:.0f} / {cat_idx.loc['Outerwear', [4,5,6,7,8]].min():.0f}",
    "Outerwear share of its year in Nov–Jan": f"{peaks.loc['Outerwear', 'Nov–Jan share of year']:.0%}",
    "Shorts / Tank Tops peak": f"{peaks.loc['Shorts', 'peak month']} {peaks.loc['Shorts', 'peak index']:.0f}, "
                               f"Nov–Feb ≈ {cat_idx.loc['Shorts', [11,12,1,2]].mean():.0f}",
    "Promo days per year (BF + BW)": f"{promo['BF days'].mean():.0f} + {promo['BW days'].mean():.0f}",
    "Promo share of annual revenue": f"{promo['promo share of year'].mean():.1%}",
    "Discount given per year (avg)": f"${promo['discount given'].mean():,.0f}",
    "Sale day vs. other Nov–Dec day": f"{promo['promo $/day ÷ other Nov–Dec $/day'].mean():.2f}×",
    "Seasonal profile stable pre-competitor?": f"correlation {store_idx.corr(pre_idx):.2f}",
}
pd.Series(summary, name="value").to_frame()